In [1]:
!pip install --upgrade torchao peft transformers -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 108.8 MB/s eta 0:00:0000:010:01


In [2]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
    stream=sys.stdout
)
log = logging.getLogger(__name__)

In [3]:
import logging
from typing import List, Dict
from datasets import load_dataset

log = logging.getLogger(__name__)

class MathDatasetBuilder:
    def __init__(self, dataset_name: str, max_prompt_len: int = 1024):
        self.dataset_name = dataset_name
        self.max_prompt_len = max_prompt_len
        self.instruction_suffix = (
            "\nPlease reason step by step, and put your final answer within \\boxed{}."
        )

    def format_chat_prompt(self, problem: str, tokenizer) -> str:
        full_problem = problem + self.instruction_suffix
        
        messages = [
            {"role": "user", "content": full_problem},
        ]
        
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _process_dataset(self, dataset_split, tokenizer) -> List[Dict]:
        """Hàm xử lý chung để encode prompt và trích xuất gold_answer."""
        out = []
        for item in dataset_split:
            problem = item.get("problem", "")
            gold_answer = item.get("answer", "")

            if not problem: continue

            txt = self.format_chat_prompt(problem, tokenizer)
            ids = tokenizer.encode(txt, add_special_tokens=False)

            if len(ids) <= self.max_prompt_len:
                out.append({
                    "prompt": txt,
                    "prompt_ids": ids,
                    "gold_answer": gold_answer
                })

        return out

    def load_train_data(self, tokenizer) -> List[Dict]:
        log.info(f"Đang load data train từ {self.dataset_name}...")
        ds = load_dataset(self.dataset_name, split="test")

        # 400 sample đầu
        train_ds = ds.select(range(400))

        out = self._process_dataset(train_ds, tokenizer)
        log.info(f"Đã load {len(out)} training examples.")
        return out

    def load_val_data(self, tokenizer) -> List[Dict]:
        log.info(f"Đang load data validation từ {self.dataset_name}...")
        ds = load_dataset(self.dataset_name, split="test")

        # 100 sample
        val_ds = ds.select(range(400, len(ds)))

        out = self._process_dataset(val_ds, tokenizer)
        log.info(f"Đã load {len(out)} validation examples.")
        return out

if __name__ == "__main__":
    # test
    from transformers import AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-math-7b-instruct", trust_remote_code=True)

    builder = MathDatasetBuilder("HuggingFaceH4/MATH-500")

    train_data = builder.load_train_data(tokenizer)
    val_data = builder.load_val_data(tokenizer)

    print("Example:")
    print("Prompt:", train_data[0]["prompt"][:200], "...") # In dài ra chút để thấy rõ phần instruction
    print("Gold Answer:", train_data[0]["gold_answer"])

07:26:59 | INFO | NumExpr defaulting to 2 threads.
07:27:00 | INFO | TensorFlow version 2.20.0 available.
07:27:00 | INFO | JAX version 0.7.2 available.
07:27:24 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:27:24 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/deepseek-math-7b-instruct/0a5828f800a36df0fd7f0ed581b983246c0677ff/config.json "HTTP/1.1 200 OK"
07:27:24 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/deepseek-ai/deepseek-math-7b-instruct/0a5828f800a36df0fd7f0ed581b983246c0677ff/config.json "HTTP/1.1 200 OK"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

07:27:25 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


07:27:25 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
07:27:25 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/deepseek-math-7b-instruct/0a5828f800a36df0fd7f0ed581b983246c0677ff/tokenizer_config.json "HTTP/1.1 200 OK"
07:27:25 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/deepseek-ai/deepseek-math-7b-instruct/0a5828f800a36df0fd7f0ed581b983246c0677ff/tokenizer_config.json "HTTP/1.1 200 OK"


...

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

07:27:27 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/MATH-500.py "HTTP/1.1 404 Not Found"
07:27:27 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceH4/MATH-500/HuggingFaceH4/MATH-500.py "HTTP/1.1 404 Not Found"
07:27:27 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceH4/MATH-500/revision/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be "HTTP/1.1 200 OK"
07:27:27 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/.huggingface.yaml "HTTP/1.1 404 Not Found"
07:27:27 | INFO | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=HuggingFaceH4/MATH-500 "HTTP/1.1 200 OK"
07:27:27 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceH4/MATH-500/tree/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/data?recursive=true&expand=

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

07:27:28 | INFO | Đã load 400 training examples.
07:27:28 | INFO | Đang load data validation từ HuggingFaceH4/MATH-500...
07:27:28 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
07:27:28 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/MATH-500/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/README.md "HTTP/1.1 200 OK"
07:27:28 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/MATH-500.py "HTTP/1.1 404 Not Found"
07:27:28 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceH4/MATH-500/HuggingFaceH4/MATH-500.py "HTTP/1.1 404 Not Found"
07:27:28 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/.huggingface.yaml "HTTP/1.1 404 Not Found"
07:27:28 |

In [4]:
import logging
import torch
from transformers import AutoModelForCausalLM

log = logging.getLogger(__name__)

def load_policy_and_ref_models(model_name: str, dtype: torch.dtype, device_map: str = "auto"):
    """Loads the trainable Actor (Policy) and frozen Reference models."""

    log.info(f"Loading Actor model (Policy): {model_name}")
    policy_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True
    )
    policy_model.gradient_checkpointing_enable()
    policy_model.train()

    log.info(f"Loading Reference model (Frozen): {model_name}")
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True
    )
    ref_model.eval()

    # Đóng băng toàn bộ trọng số của Reference Model
    for param in ref_model.parameters():
        param.requires_grad_(False)

    return policy_model, ref_model

In [5]:
import re
import torch
import logging
from transformers import AutoTokenizer, AutoModel

log = logging.getLogger(__name__)

class RuleBasedRewardScorer:
    def __init__(self, device: torch.device):
        self.device = device
        log.info("Đang khởi tạo Rule-Based Reward Scorer...")

    def extract_boxed_answer(self, text: str) -> str:

        # tìm pattern \boxed{}
        matches = re.findall(r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', text)
        if matches:
            return matches[-1].strip()

        # fallback
        fallback = re.findall(r'The answer is:?\s*\$?([^\$\n]+)\$?', text, re.IGNORECASE)
        if fallback:
            return fallback[-1].strip()

        return ""

    def normalize_answer(self, ans: str) -> str:
        if not ans: return ""
        ans = ans.replace(" ", "").lower().replace(",", "")
        ans = ans.rstrip('.')
        return ans

    def get_scores(self, responses: list[str], gold_answer: str) -> torch.Tensor:
        """
        - Đúng đáp án: +1.0 (Phần thưởng tối đa)
        - Có \boxed{} nhưng tính sai: -0.5 (Phạt nhẹ)
        - Không đưa ra được kết luận / Lạc đề: -1.0 (Phạt nặng)
        """
        rewards = []
        norm_gold = self.normalize_answer(gold_answer)

        for resp in responses:
            pred_ans = self.extract_boxed_answer(resp)

            if not pred_ans:
                rewards.append(-1.0)
                continue

            norm_pred = self.normalize_answer(pred_ans)

            if norm_pred == norm_gold:
                rewards.append(1.0)
            else:
                rewards.append(-0.5)

        return torch.tensor(rewards, dtype=torch.float32, device=self.device)


class RewardModelScorer:
    def __init__(self, rm_name: str, dtype: torch.dtype, device: str):
        self.device = device
        log.info(f"Đang khởi tạo Reward Model {rm_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            rm_name,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.model = AutoModel.from_pretrained(
            rm_name,
            torch_dtype=dtype,
            trust_remote_code=True,
            device_map=device
        )
        self.model.eval()

    def get_reward(self, questions: list[str], responses: list[str]) -> torch.Tensor:

        rewards = []
        for q, r in zip(questions, responses):
            messages = [
                {"role": "user", "content": q},
                {"role": "assistant", "content": r}
            ]

            # Format
            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)

                if hasattr(outputs, "score"):
                    score = outputs.score[0].item()
                elif hasattr(outputs, "logits"):
                    score = outputs.logits[0, 0].item()
                else:
                    # Fallback
                    score = outputs[0][0].item()

            rewards.append(score)

        return torch.tensor(rewards, dtype=torch.float32, device=self.device)

In [6]:
import numpy as np
from typing import List

def compute_lpro_advantages(
    rewards: List[float],
    lengths: List[int],
    lambda_len: float = 0.10,
    eps: float = 1e-8
) -> np.ndarray:

    r = np.array(rewards, dtype=np.float64)
    L = np.array(lengths, dtype=np.float64)

    # Z-score chuẩn hóa
    z_r = (r - r.mean()) / (r.std() + eps)
    z_L = (L - L.mean()) / (L.std() + eps)

    advantages = z_r - lambda_len * z_L
    return advantages

In [7]:
import torch
from typing import Tuple

def compute_dapo_token_loss(
    new_logprobs: torch.Tensor,
    old_logprobs: torch.Tensor,
    advantage: float,
    mask: torch.Tensor,
    eps_low: float = 0.20,
    eps_high: float = 0.28
) -> Tuple[torch.Tensor, int]:

    ratio = torch.exp(new_logprobs - old_logprobs)
    ratio_clipped = torch.clamp(ratio, 1.0 - eps_low, 1.0 + eps_high)

    adv_tensor = torch.full_like(new_logprobs, advantage)
    surrogate1 = ratio * adv_tensor
    surrogate2 = ratio_clipped * adv_tensor

    # Pessimistic bound (min) và đổi dấu vì PyTorch dùng Gradient Descent (minimize)
    token_losses = -torch.min(surrogate1, surrogate2)

    # Chỉ tính loss trên các token không phải padding
    masked_loss = token_losses * mask

    return masked_loss.sum(), mask.sum().item()

In [8]:
import os
import math
import random
import logging
import torch
import numpy as np
from torch.optim import AdamW
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

# from nexus.models.policy import load_policy_and_ref_models
# from nexus.models.reward import RewardModelScorer, RuleBasedRewardScorer
# from nexus.rl.advantages import compute_lpro_advantages
# from nexus.rl.loss import compute_dapo_token_loss
# from nexus.data.builder import MathDatasetBuilder

log = logging.getLogger(__name__)

class NexusTrainer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.bfloat16 if cfg.bf16 and torch.cuda.is_bf16_supported() else torch.float32
        
        self._set_seed()
        self._setup_components()

    def _set_seed(self):
        random.seed(self.cfg.seed)
        np.random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)

    def _setup_components(self):
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_name, trust_remote_code=True)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Models
        self.model, self.ref_model = load_policy_and_ref_models(self.cfg.model_name, self.dtype)
        if getattr(self.cfg, "use_lora", False):
            log.info("Finetune LoRA: Đang cấu hình PEFT/LoRA adapter...")
            
            lora_config = LoraConfig(
                task_type=TaskType.CAUSAL_LM,
                inference_mode=False,
                r=self.cfg.lora_r,
                lora_alpha=self.cfg.lora_alpha,
                lora_dropout=self.cfg.lora_dropout,
                target_modules=self.cfg.lora_target_modules
            )
            
            self.model = get_peft_model(self.model, lora_config)
            self.model.print_trainable_parameters()
            
            if hasattr(self.model, "enable_input_require_grads"):
                self.model.enable_input_require_grads()
        else:
            log.info("Full Finetune: Đang cấu hình để fine-tune toàn bộ model...")

        # Setup Reward Scorer
        if getattr(self.cfg, "use_rule_based_rm", True):
            self.rm_scorer = RuleBasedRewardScorer(self.device)
        else:
            self.rm_scorer = RewardModelScorer(self.cfg.rm_model_name, self.dtype, self.device)

    @staticmethod
    def get_resp_log_probs(model, full_ids: torch.Tensor, prompt_len: int, no_grad: bool = False) -> torch.Tensor:
        ctx = torch.no_grad() if no_grad else torch.enable_grad()
        with ctx:
            out = model(input_ids=full_ids)
            lp = torch.log_softmax(out.logits[0], dim=-1) # (seq_len, vocab_size)
            resp_ids = full_ids[0, prompt_len:]
            resp_lp = lp[prompt_len - 1 : full_ids.shape[1] - 1] # (resp_len, vocab_size)
            return resp_lp.gather(1, resp_ids.unsqueeze(-1)).squeeze(-1) # (resp_len,) # log-prob của token tiếp theo

    # eval per epoch
    def evaluate(self, val_dataset, batch_size: int = 16):
        self.model.eval()
        correct = 0
        total = len(val_dataset)
        batch_size = self.cfg.eval_batch_size if hasattr(self.cfg, "eval_batch_size") else batch_size
        log.info(f"Đang evaluation trên {total} samples với batch_size = {batch_size}...")
        scorer = self.rm_scorer if isinstance(self.rm_scorer, RuleBasedRewardScorer) else RuleBasedRewardScorer(self.device)
        
        for i in tqdm(range(0, total, batch_size), desc="Evaluating"):
            batch_examples = val_dataset[i : i + batch_size]
            
            batch_prompts = [example["prompt"] for example in batch_examples]
            batch_gold_answers = [example.get("gold_answer", "") for example in batch_examples]
            
            inputs = self.tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.cfg.max_prompt_len
            ).to(self.device)
            
            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]
            
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=self.cfg.bf16):
                    gen_out = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=self.cfg.max_new_tokens,
                        do_sample=False,
                        pad_token_id=self.tokenizer.eos_token_id,
                    )
            
            for j, example in enumerate(batch_examples):
                # Do Left-Padding, độ dài của phần prompt trong ma trận đầu vào 
                # của tất cả các câu đều được đưa về bằng bằng input_ids.shape[1].
                # Vì vậy, phần token mới sinh ra (response) sẽ bắt đầu chính xác từ index này trở đi.
                resp_ids = gen_out[j, input_ids.shape[1]:]
                
                # Loại bỏ các token kết thúc (eos) hoặc token đệm dư thừa ở đuôi câu trả lời
                pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
                actual_len = max(pad_mask.sum().item(), 1)
                
                # Giải mã tensor số thành văn bản chữ
                resp_text = self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True)
                
                # Trích xuất thẻ \boxed{} và đối chiếu kết quả
                pred_ans = scorer.extract_boxed_answer(resp_text)
                if scorer.normalize_answer(pred_ans) == scorer.normalize_answer(batch_gold_answers[j]) and batch_gold_answers[j]:
                    correct += 1
            
            # Giải phóng bộ nhớ đệm GPU sau mỗi batch
            del inputs, gen_out
            torch.cuda.empty_cache()
            
        acc = (correct / total) * 100
        log.info(f"Evaluated. Accuracy = {acc:.2f}% ({correct}/{total})")
        
        self.model.train()
        return acc

    # training loop
    def train(self, prompt_batch_size: int = 16):
        prompt_batch_size = self.cfg.train_batch_size if hasattr(self.cfg, "train_batch_size") else prompt_batch_size
        dataset_builder = MathDatasetBuilder(self.cfg.dataset_name, self.cfg.max_prompt_len)
        
        train_dataset = dataset_builder.load_train_data(self.tokenizer)
        val_dataset = dataset_builder.load_val_data(self.tokenizer)

        optimizer = AdamW(self.model.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        
        total_batches = math.ceil(len(train_dataset) / prompt_batch_size)
        total_steps = self.cfg.num_epochs * total_batches
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(self.cfg.warmup_ratio * total_steps), total_steps)
        
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        global_step, acc_loss, acc_reward = 0, 0.0, 0.0

        for epoch in range(self.cfg.num_epochs):
            random.shuffle(train_dataset)
            log.info(f"Epoch {epoch + 1}/{self.cfg.num_epochs} - Tổng số batches: {total_batches}, Batch size: {prompt_batch_size}")
            
            pbar = tqdm(range(0, len(train_dataset), prompt_batch_size), desc=f"Epoch {epoch + 1}/{self.cfg.num_epochs}")

            for step_idx in pbar:
                batch_examples = train_dataset[step_idx : step_idx + prompt_batch_size]
                B = len(batch_examples)
                
                prompts = [ex["prompt"] for ex in batch_examples]
                gold_answers = [ex.get("gold_answer", "") for ex in batch_examples]

                inputs = self.tokenizer(
                    prompts, 
                    return_tensors="pt", 
                    padding=True, 
                    truncation=True, 
                    max_length=self.cfg.max_prompt_len
                ).to(self.device)
                
                # [B, seq_len] -> [BxG, seq_len]
                input_ids = inputs["input_ids"].repeat_interleave(self.cfg.G, dim=0)
                attention_mask = inputs["attention_mask"].repeat_interleave(self.cfg.G, dim=0)
                prompt_len = input_ids.shape[1]

                self.model.eval()
                with torch.no_grad():
                    with torch.amp.autocast('cuda', enabled=self.cfg.bf16):
                        gen_out = self.model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            max_new_tokens=self.cfg.max_new_tokens,
                            temperature=self.cfg.temperature,
                            do_sample=True,
                            pad_token_id=self.tokenizer.eos_token_id,
                        )
                self.model.train()

                lengths, masks, full_seqs, resp_texts = [], [], [], []
                total_generated = B * self.cfg.G
                
                for i in range(total_generated):
                    resp_ids = gen_out[i, prompt_len:]
                    pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
                    actual_len = max(pad_mask.sum().item(), 1)
                    
                    lengths.append(actual_len)
                    masks.append(pad_mask.to(self.device))
                    full_seqs.append(gen_out[i])
                    resp_texts.append(self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True))

                # reward & advantage per group
                all_advs = []
                batch_mean_reward = 0.0
                
                for b in range(B):
                    # G responses per prompt
                    start_idx = b * self.cfg.G
                    end_idx = start_idx + self.cfg.G
                    group_texts = resp_texts[start_idx : end_idx]
                    gold = gold_answers[b]
                    
                    if isinstance(self.rm_scorer, RuleBasedRewardScorer):
                        rewards = self.rm_scorer.get_scores(group_texts, gold)
                    else:
                        rewards = self.rm_scorer.get_reward([prompts[b]] * self.cfg.G, group_texts)
                        
                    if not isinstance(rewards, torch.Tensor):
                        rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device)
                        
                    batch_mean_reward += rewards.mean().item()

                    if torch.all(rewards == rewards[0]):
                        all_advs.extend([0.0] * self.cfg.G)
                    else:
                        group_lens = lengths[start_idx : end_idx]
                        advs = compute_lpro_advantages(rewards.tolist(), group_lens, self.cfg.lambda_len)
                        all_advs.extend(advs.tolist())
                        
                batch_mean_reward /= B

                # loss + backprop
                total_loss_sum = torch.tensor(0.0, device=self.device)
                total_n_tokens = 0

                for i in range(total_generated):
                    if all_advs[i] == 0.0:
                        continue
                        
                    seq = full_seqs[i].unsqueeze(0).to(self.device)[:, :prompt_len + lengths[i]]
                    mask_i = masks[i][:lengths[i]]

                    with torch.amp.autocast('cuda', enabled=self.cfg.bf16):
                        new_lp = self.get_resp_log_probs(self.model, seq, prompt_len, no_grad=False)
                        old_lp = self.get_resp_log_probs(self.ref_model, seq, prompt_len, no_grad=True)

                    loss_sum, n_valid = compute_dapo_token_loss(
                        new_lp, old_lp.detach(), float(all_advs[i]), mask_i, self.cfg.eps_low, self.cfg.eps_high
                    )
                    total_loss_sum += loss_sum
                    total_n_tokens += n_valid

                if total_n_tokens == 0: 
                    del gen_out, inputs, input_ids
                    torch.cuda.empty_cache()
                    continue
                
                # loss averaged over all tokens in the batch
                loss = total_loss_sum / total_n_tokens
                loss.backward()

                acc_loss += loss.item()
                acc_reward += batch_mean_reward

                pbar.set_postfix({
                    "loss": f"{loss.item():.4f}", 
                    "reward": f"{batch_mean_reward:.2f}"
                })

                # optimizer
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # Logging & Saving
                if global_step % self.cfg.log_steps == 0:
                    log.info(f"Step {global_step} | Loss: {acc_loss/self.cfg.log_steps:.4f} | RM Score: {acc_reward/self.cfg.log_steps:.3f}")
                    acc_loss, acc_reward = 0.0, 0.0

                if global_step % self.cfg.save_steps == 0:
                    self.model.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))
                    self.tokenizer.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))

                del gen_out, inputs, input_ids, seq, new_lp, old_lp, loss_sum, total_loss_sum
                torch.cuda.empty_cache()
            
            log.info(f"Hoàn thành Train Epoch {epoch + 1}. Bắt đầu Eval...")
            self.evaluate(val_dataset)

        log.info("Training hoàn tất. Đang lưu model...")
        
        # save
        self.model.save_pretrained(self.cfg.output_dir)
        self.tokenizer.save_pretrained(self.cfg.output_dir)

        # push to hub
        if getattr(self.cfg, "push_to_hub", False):
            if not self.cfg.hf_token:
                log.error("Chưa cung cấp hf_token trong Config.")
            else:
                log.info(f"Đang đẩy model lên Hugging Face Hub ({self.cfg.hub_repo_id})...")
                try:
                    self.model.push_to_hub(
                        self.cfg.hub_repo_id, 
                        token=self.cfg.hf_token,
                        commit_message="Upload Nexus Qwen-Math weights"
                    )
                    self.tokenizer.push_to_hub(
                        self.cfg.hub_repo_id, 
                        token=self.cfg.hf_token,
                        commit_message="Upload tokenizer"
                    )
                    log.info("Đã đẩy model lên Hugging Face Hub thành công!")
                except Exception as e:
                    log.error(f"Có lỗi xảy ra khi đẩy lên Hub: {e}")

07:31:29 | WARNING | Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
07:31:29 | WARNING | Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so


In [ ]:
import json
import logging
import os

# from nexus.nexus_trainer.trainer import NexusTrainer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

class Config:
    def __init__(self):
        self.model_name = "deepseek-ai/deepseek-math-7b-instruct"
        self.rm_model_name = "Qwen/Qwen2.5-Math-RM-72B"
        self.output_dir = "./nexus-7b"
        self.hub_repo_id = "YOUR_HF_USERNAME/nexus-7b"

        self.dataset_name = "HuggingFaceH4/MATH-500"
        self.max_prompt_len = 512
        self.train_batch_size = 2
        self.eval_batch_size = 16

        self.G = 4
        self.max_new_tokens = 2048
        self.temperature = 0.7
        self.top_p = 0.95

        self.eps_low = 0.20
        self.eps_high = 0.28
        self.lambda_len = 0.05
        self.eps_r = 1e-8
        self.eps_l = 1e-8

        self.use_lora = True
        self.use_rule_based_rm = True

        self.lora_r = 16
        self.lora_alpha = 32
        self.lora_dropout = 0.05
        self.lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

        self.num_epochs = 5

        self.lr = 2e-4
        self.weight_decay = 1e-2
        self.warmup_ratio = 0.05
        self.grad_clip = 1.0
        self.bf16 = True

        self.log_steps = 10
        self.save_steps = 100
        self.seed = 42

        self.push_to_hub = True
        self.hf_token = ""


def main():
    cfg = Config()

    cfg_dict = {k: v for k, v in cfg.__dict__.items() if not k.startswith('__')}
    log.info("Starting training with config:\n" + json.dumps(cfg_dict, indent=2, default=str))

    trainer = NexusTrainer(cfg)
    trainer.train()

if __name__ == "__main__":
    main()